# Gate 2 · e09 — CRNN tuned with Optuna for wheeze recall (Colab GPU)

**Goal.** The e06–e08 confusion matrices showed that the **CRNN with a middle number of parameters (~0.3–0.5 M)** is the best compromise for the W-vs-FC step. Here that architecture is tuned with **Optuna**,
**prioritising wheeze (W) recall without neglecting fine-crackle (FC) performance**, evaluated on the official fold (train 2022–2024, test 2025), compared with the e06 CRNN and with the state of the art, and exported to **ONNX**.

**How the priority is built.** For every candidate the decision threshold is chosen on the *validation* patients (never on 2025) to maximise
`score = 0.65 · recall_W + 0.35 · recall_FC − 2 · max(0, 0.80 − recall_FC)`: wheeze recall counts almost twice as much as crackle recall, but a model whose FC recall falls under 0.80 is penalised.
The loss also has a searchable FC weight (`w_pos_weight_factor` < 1 makes missing a wheeze more expensive). The test year 2025 is used only for the final report.

**Steps.** 1 options → 2 setup (GPU check) → 3 data → 4 model + search space → 5 Optuna search (live plot) → 6 search diagnostics → 7 final training of e06-CRNN and tuned CRNN (training curves shown here) → 8 test results, confusion matrices, CIs →
9 comparison with the state of the art → 10 ONNX export + check → 11 model report and architecture drawing → 12 zip.
No Weights & Biases: everything is displayed in the notebook and saved to `/content/outputs`. **It needs a CUDA GPU** (Runtime → Change runtime type → T4); start with `PRESET = "smoke"`.

In [ ]:
# ── Options ──────────────────────────────────────────────────────────────────────────────────────────
from pathlib import Path

PRESET = "standard"      # "smoke" (minutes: checks everything) | "standard" | "long"
PRESETS = {
    "smoke":    dict(N_TRIALS=4,   TIMEOUT_MIN=20,  MAX_EPOCHS=3,  FINAL_SEEDS=[0, 1],    FINAL_EPOCHS=3),
    "standard": dict(N_TRIALS=40,  TIMEOUT_MIN=120, MAX_EPOCHS=25, FINAL_SEEDS=[0, 1, 2], FINAL_EPOCHS=40),
    "long":     dict(N_TRIALS=100, TIMEOUT_MIN=240, MAX_EPOCHS=30, FINAL_SEEDS=[0, 1, 2, 3, 4], FINAL_EPOCHS=40),
}
N_TRIALS, TIMEOUT_MIN, MAX_EPOCHS, FINAL_SEEDS, FINAL_EPOCHS = (PRESETS[PRESET][k] for k in ("N_TRIALS", "TIMEOUT_MIN", "MAX_EPOCHS", "FINAL_SEEDS", "FINAL_EPOCHS"))

HELD_OUT = "2025"                  # official test year: train 2022-2024, test 2025 (never used to choose anything)
VAL_FRAC, VAL_SEED = 0.20, 0       # validation = 20 % of the training PATIENTS (fixed split for the search and the final models)
PARAM_MIN, PARAM_MAX = 250_000, 700_000     # "middle" size: trials outside this range are rejected before training
W_WEIGHT, FC_FLOOR = 0.65, 0.80    # priority: score = W_WEIGHT*recall_W + (1-W_WEIGHT)*recall_FC - 2*max(0, FC_FLOOR - recall_FC)
PATIENCE, FINAL_PATIENCE = 5, 8
THRESHOLD = 0.5                    # reference threshold (the tuned one is chosen on validation data)
N_BOOT = 500                       # patient-clustered bootstrap resamples for the confidence intervals
REQUIRE_GPU = True                 # stop if CUDA is not available (False only for a CPU test)

DATA_ROOT = Path("/content/data/raw_data")
OUT_DIR = Path("/content/outputs")

In [ ]:
# ── Setup + GPU check ────────────────────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile", "optuna", "onnx", "onnxruntime"], check=False)

import json, os, shutil, time, warnings
from math import gcd
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
import soundfile as sf
import optuna
from IPython.display import display, clear_output
from joblib import Parallel, delayed
from scipy import signal
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU: Runtime > Change runtime type > Hardware accelerator = GPU (T4), then reconnect and run again.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "(CPU: test mode)")
if DEVICE.type == "cuda":
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"preset {PRESET}: {N_TRIALS} trials (max {TIMEOUT_MIN} min), {MAX_EPOCHS} epochs per trial, final seeds {FINAL_SEEDS}, params {PARAM_MIN:,}-{PARAM_MAX:,}")

## 3 · Data: download SPRSound (train 2022–2024, test 2025) and extract the W / FC events as log-mel

In [ ]:
import shutil

def download_sprsound(root: Path):
    """Clone SPRSound and copy the recordings to root/{train,test}/{wav,json}, prefixing the file names with the year (as get_data.sh)."""
    work = Path("/content/SPRSound")
    if not work.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SJTU-YONGFU-RESEARCH-GRP/SPRSound.git", str(work)], check=True)
    sources = [("BioCAS2022/train2022_wav", "BioCAS2022/train2022_json", "train", "2022"),
               ("BioCAS2022/test2022_wav", "BioCAS2022/test2022_json", "train", "2022"),
               ("BioCAS2023/test2023_wav", "BioCAS2023/test2023_json", "train", "2023"),
               ("BioCAS2024/test2024_wav", "BioCAS2024/test2024_json", "train", "2024"),
               ("BioCAS2025/test2025_wav", "BioCAS2025/test2025_json", "test", "2025")]
    for wav_dir, json_dir, split, year in sources:
        if not (work / wav_dir).is_dir():
            print("not found, skipped:", wav_dir)
            continue
        (root / split / "wav").mkdir(parents=True, exist_ok=True)
        (root / split / "json").mkdir(parents=True, exist_ok=True)
        jsons = {p.name: p for p in (work / json_dir).rglob("*.json")}
        n = 0
        for wav in sorted((work / wav_dir).glob("*.wav")):
            js = jsons.get(wav.stem + ".json")
            if js is None:
                continue
            shutil.copy2(wav, root / split / "wav" / f"{year}__{wav.name}")
            shutil.copy2(js, root / split / "json" / f"{year}__{js.name}")
            n += 1
        print(f"{split:5s} {year}: {n} recordings")

if not list((DATA_ROOT / "train" / "wav").glob("*.wav")):
    download_sprsound(DATA_ROOT)
print({s: len(list((DATA_ROOT / s / "wav").glob("*.wav"))) for s in ("train", "test")})

In [ ]:
SR, WIN_S, N_FFT, HOP, N_MELS, FMIN, FMAX = 4000, 3.0, 512, 128, 64, 50.0, 2000.0
CLASSES = ["W", "FC"]
EVENT_TYPES = {"wheeze": 0, "fine crackle": 1}

def mel_filterbank():
    freqs = np.fft.rfftfreq(N_FFT, 1 / SR)
    mel = lambda f: 2595 * np.log10(1 + f / 700)
    inv = lambda m: 700 * (10 ** (m / 2595) - 1)
    pts = inv(np.linspace(mel(FMIN), mel(FMAX), N_MELS + 2))
    fb = np.zeros((N_MELS, len(freqs)))
    for i in range(N_MELS):
        lo, c, hi = pts[i], pts[i + 1], pts[i + 2]
        fb[i] = np.maximum(0, np.minimum((freqs - lo) / (c - lo), (hi - freqs) / (hi - c)))
    return fb

FB = mel_filterbank()

def to_mel(seg):
    n = int(WIN_S * SR)
    w = seg[(len(seg) - n) // 2:(len(seg) - n) // 2 + n] if len(seg) >= n else np.tile(seg, int(np.ceil(n / len(seg))))[:n]
    _, _, z = signal.stft(w, nperseg=N_FFT, noverlap=N_FFT - HOP, window="hann", boundary="even", padded=True)
    d = 10 * np.log10(np.maximum(FB @ (np.abs(z) ** 2), 1e-10))
    return ((d - d.mean()) / (d.std() + 1e-6)).astype(np.float32)

def recording_events(wav, js, year, patient):
    ann = json.loads(Path(js).read_text(encoding="utf-8"))
    ev = []
    for e in ann.get("event_annotation", []):
        label = EVENT_TYPES.get(str(e.get("type", "")).strip().lower())
        try:
            s, t = int(e["start"]), int(e["end"])
        except (KeyError, ValueError):
            continue
        if label is not None and t - s >= 20:
            ev.append((label, s, t))
    if not ev:
        return []
    x, sr = sf.read(str(wav), dtype="float32", always_2d=True)
    x = x[:, 0]
    if sr != SR:
        g = gcd(int(sr), SR)
        x = signal.resample_poly(x, SR // g, sr // g).astype(np.float32)
    out = []
    for label, s, t in ev:
        seg = x[int(s * SR / 1000):int(t * SR / 1000)]
        if len(seg) >= 64:
            out.append((to_mel(seg), label, year, patient))
    return out

def load_real_events(root):
    jobs = []
    for split in ("train", "test"):
        for wav in sorted((root / split / "wav").glob("*.wav")):
            js = root / split / "json" / f"{wav.stem}.json"
            year, rest = wav.stem.split("__", 1)
            pid = rest.split("_")[0]
            if js.exists():
                jobs.append((wav, js, year, str(float(pid)) if pid.isdigit() else "nan"))
    rows = [r for rs in Parallel(n_jobs=-1)(delayed(recording_events)(*j) for j in jobs) for r in rs]
    X = np.stack([r[0] for r in rows])
    y, year, patient = (np.array([r[i] for r in rows]) for i in (1, 2, 3))
    return X, y, year, patient

cache = OUT_DIR / "real_events.npz"
if cache.exists():
    z = np.load(cache, allow_pickle=True); X, y, year, patient = z["X"], z["y"], z["year"], z["patient"]
else:
    X, y, year, patient = load_real_events(DATA_ROOT)
    np.savez(cache, X=X, y=y, year=year, patient=patient)
tab = pd.crosstab(year, np.array(CLASSES)[y]).rename_axis("year")
print(f"{len(y)} events"); display(tab)

## 4 · The CRNN, its search space and the training function

`CRNN` = three Conv3×3-BN-ReLU-MaxPool blocks (the frequency axis is reduced, the time axis is kept), a **bidirectional GRU** over the time steps, pooling over time, and a small MLP head that outputs the logit of "fine crackle"
(label 1; wheeze = 0). The e06 CRNN is the point `(c1=16, ratio=2, hidden=96, 1 layer, mean pooling, dropout 0.3, 64 units)`; it is enqueued as the first trial so Optuna starts from it.

In [ ]:
def conv_bn(c_in, c_out):
    return [nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True)]

def round4(v):
    return max(4, int(round(v / 4)) * 4)

class CRNN(nn.Module):
    def __init__(self, ch=(16, 32, 64), hidden=96, layers=1, pool="mean", drop=0.3, units=64):
        super().__init__()
        blocks, c_in = [], 1
        for i, c in enumerate(ch):
            blocks += conv_bn(c_in, c) + [nn.MaxPool2d(2 if i < 2 else (2, 1))]      # 4 modules per block
            c_in = c
        self.conv = nn.Sequential(*blocks)
        self.rnn = nn.GRU(ch[-1] * (N_MELS // 8), hidden, num_layers=layers, batch_first=True, bidirectional=True)
        self.pool = pool
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(2 * hidden, units), nn.ReLU(), nn.Linear(units, 1))
    def forward(self, x):
        h = self.conv(x).permute(0, 3, 1, 2).flatten(2)          # (B, T', C*F')
        o = self.rnn(h)[0]
        o = o.mean(1) if self.pool == "mean" else o.max(1).values
        return self.head(o).squeeze(1)

def channels_of(P):
    return (P["c1"], round4(P["c1"] * P["ratio"]), round4(P["c1"] * P["ratio"] ** 2))

def build_model(P):
    return CRNN(channels_of(P), P["hidden"], P["gru_layers"], P["pool"], P["dropout"], P["units"])

def count_params(m):
    return sum(p.numel() for p in m.parameters())

# e06 CRNN (reference) = default point of the search space
BASELINE_P = dict(c1=16, ratio=2.0, hidden=96, gru_layers=1, pool="mean", dropout=0.3, units=64, lr=3e-4, wd=1e-3, batch=64, w_pos_weight_factor=1.0)

def suggest(trial):
    return dict(
        c1=trial.suggest_categorical("c1", [8, 12, 16, 24, 32]),
        ratio=trial.suggest_categorical("ratio", [1.5, 2.0]),
        hidden=trial.suggest_categorical("hidden", [32, 48, 64, 96, 128, 160]),
        gru_layers=trial.suggest_int("gru_layers", 1, 2),
        pool=trial.suggest_categorical("pool", ["mean", "max"]),
        dropout=trial.suggest_float("dropout", 0.0, 0.5),
        units=trial.suggest_categorical("units", [32, 64, 128]),
        lr=trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        wd=trial.suggest_float("wd", 1e-5, 1e-2, log=True),
        batch=trial.suggest_categorical("batch", [32, 64, 128]),
        w_pos_weight_factor=trial.suggest_float("w_pos_weight_factor", 0.3, 1.5),   # weight of the FC class relative to the balanced loss; < 1 favours wheeze recall
    )

assert build_model(BASELINE_P)(torch.zeros(2, 1, N_MELS, 95)).shape == (2,)
print(f"e06 CRNN reference: {count_params(build_model(BASELINE_P)):,} parameters; allowed range {PARAM_MIN:,}-{PARAM_MAX:,}")

In [ ]:
# ── Priority score, threshold selection, data on the GPU, augmentation, prediction, training ─────────
def rec_at(y_true, p, thr):
    pred = p >= thr
    return float((~pred[y_true == 0]).mean()), float(pred[y_true == 1].mean())           # recall W, recall FC

def priority(rW, rFC):
    return W_WEIGHT * rW + (1 - W_WEIGHT) * rFC - 2 * max(0.0, FC_FLOOR - rFC)

def best_threshold(y_true, p):
    """Threshold that maximises the priority score on (validation) data. Returns (threshold, score, recall_W, recall_FC)."""
    best = (0.5, -9, 0, 0)
    for t in np.unique(np.quantile(p, np.linspace(0.01, 0.99, 197))):
        rW, rFC = rec_at(y_true, p, t); s = priority(rW, rFC)
        if s > best[1]:
            best = (float(t), s, rW, rFC)
    return best

# fixed patient-level split of the training years; the held-out year is the test set
tr = year != HELD_OUT
Xtr, ytr, gtr = X[tr], y[tr], patient[tr]
tr_i, va_i = next(GroupShuffleSplit(1, test_size=VAL_FRAC, random_state=VAL_SEED).split(Xtr, ytr, gtr))
to_g = lambda a: torch.from_numpy(np.ascontiguousarray(a))[:, None].to(DEVICE)
Xa, ya, ya_np = to_g(Xtr[tr_i]), torch.from_numpy(ytr[tr_i].astype(np.float32)).to(DEVICE), ytr[tr_i]
Xv, yv = to_g(Xtr[va_i]), ytr[va_i]
Xe, ye, ge = to_g(X[~tr]), y[~tr], patient[~tr]
print(f"train {len(ya_np)} (W {(ya_np == 0).sum()}, FC {(ya_np == 1).sum()}) · validation {len(yv)} (W {(yv == 0).sum()}, FC {(yv == 1).sum()}) · test {HELD_OUT} {len(ye)} (W {(ye == 0).sum()}, FC {(ye == 1).sum()})")

def augment(x):
    """GPU SpecAugment-like: circular time shift, one frequency mask (<= 8 bins), one time mask (<= 10 frames), per sample."""
    B, _, F, T = x.shape
    dev = x.device
    shift = torch.randint(0, T, (B,), device=dev)
    idx = (torch.arange(T, device=dev)[None] - shift[:, None]) % T
    x = x.gather(3, idx[:, None, None, :].expand(B, 1, F, T))
    f = torch.randint(0, 9, (B,), device=dev); f0 = (torch.rand(B, device=dev) * (F - f)).long()
    ar = torch.arange(F, device=dev)[None]
    x = x.masked_fill(((ar >= f0[:, None]) & (ar < (f0 + f)[:, None]))[:, None, :, None], 0)
    t = torch.randint(0, 11, (B,), device=dev); t0 = (torch.rand(B, device=dev) * (T - t)).long()
    ar = torch.arange(T, device=dev)[None]
    return x.masked_fill(((ar >= t0[:, None]) & (ar < (t0 + t)[:, None]))[:, None, None, :], 0)

@torch.no_grad()
def predict(model, Xg, bs=512):
    model.eval()
    out = []
    for i in range(0, len(Xg), bs):
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            out.append(torch.sigmoid(model(Xg[i:i + bs]).float()))
    return torch.cat(out).cpu().numpy()

def fit(P, seed, max_epochs, patience, trial=None):
    """Train one CRNN. Early stopping on validation AUC; returns the best-epoch model, the per-epoch history and validation / test probabilities of FC."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = build_model(P).to(DEVICE)
    scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")
    opt = torch.optim.AdamW(model.parameters(), lr=P["lr"], weight_decay=P["wd"])
    pos_w = torch.tensor(float((ya_np == 0).sum() / max((ya_np == 1).sum(), 1)) * P["w_pos_weight_factor"], device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    hist, best = [], dict(auc=-1.0, bad=0, w=None)
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(len(Xa), device=DEVICE)
        tot, nb = 0.0, 0
        for i in range(0, len(perm), P["batch"]):
            b = perm[i:i + P["batch"]]
            if len(b) < 4:
                continue
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss = loss_fn(model(augment(Xa[b])).float(), ya[b])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += float(loss.detach()); nb += 1
        pv = predict(model, Xv)
        auc = roc_auc_score(yv, pv)
        _, sc, rW, rFC = best_threshold(yv, pv)
        hist.append(dict(epoch=ep + 1, train_loss=tot / max(nb, 1), val_auc=auc, val_score=sc, val_recall_W=rW, val_recall_FC=rFC))
        if auc > best["auc"] + 1e-4:
            best.update(auc=auc, bad=0, w={k: v.detach().clone() for k, v in model.state_dict().items()})
        else:
            best["bad"] += 1
        if trial is not None:
            trial.report(auc, ep)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if best["bad"] >= patience:
            break
    model.load_state_dict(best["w"])
    pv, pe = predict(model, Xv), predict(model, Xe)
    return dict(model=model, hist=pd.DataFrame(hist), pv=pv, pe=pe, val_auc=best["auc"], epochs=len(hist), seed=seed)

## 5 · Optuna search

TPE sampler + median pruner (a trial is stopped early when its validation AUC is below the median of earlier trials at the same epoch). The study is stored in SQLite (`/content/outputs/optuna_crnn.db`), so running the cell again **resumes** it.
Trials outside the parameter range are rejected before training (they do not count). A live plot of the objective is drawn every few trials.

In [ ]:
def objective(trial):
    P = suggest(trial)
    n = count_params(build_model(P))
    trial.set_user_attr("params", n)
    if not PARAM_MIN <= n <= PARAM_MAX:
        trial.set_user_attr("reason", "params out of range")
        raise optuna.TrialPruned()
    try:
        r = fit(P, seed=0, max_epochs=MAX_EPOCHS, patience=PATIENCE, trial=trial)
    finally:
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    thr, sc, rW, rFC = best_threshold(yv, r["pv"])
    trial.set_user_attr("val_auc", r["val_auc"]); trial.set_user_attr("val_recall_W", rW); trial.set_user_attr("val_recall_FC", rFC); trial.set_user_attr("epochs", r["epochs"])
    del r
    return sc

study = optuna.create_study(study_name=f"crnn_{PRESET}", storage=f"sqlite:///{OUT_DIR / 'optuna_crnn.db'}", load_if_exists=True, direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=42, multivariate=True, n_startup_trials=min(10, max(2, N_TRIALS // 4))),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=min(8, max(2, N_TRIALS // 5)), n_warmup_steps=4))
if not study.trials:
    study.enqueue_trial(BASELINE_P)                       # start from the e06 CRNN
S = optuna.trial.TrialState
real = lambda t: t.state in (S.COMPLETE, S.PRUNED) and t.user_attrs.get("reason") is None      # trials that were really trained
t_search = time.time()

def progress(study, trial):
    done = [t for t in study.trials if real(t)]
    comp = [t for t in study.trials if t.state == S.COMPLETE]
    msg = (f"trial {trial.number:3d} {trial.state.name:8s} " + (f"score {trial.value:.4f} " if trial.value is not None else "") +
           f"params {trial.user_attrs.get('params', 0):>7,} | trained {len(done)}/{N_TRIALS} · complete {len(comp)} | best {study.best_value if comp else float('nan'):.4f} | {(time.time() - t_search) / 60:.0f} min")
    if trial.user_attrs.get("reason") is None:
        print(msg, flush=True)
    if comp and len(done) % 5 == 0 and trial.user_attrs.get("reason") is None:
        vals = [t.value for t in comp]
        fig, ax = plt.subplots(figsize=(7, 2.6))
        ax.plot([t.number for t in comp], vals, "o", alpha=.6, label="trial"); ax.plot([t.number for t in comp], np.maximum.accumulate(vals), "r-", label="best so far")
        ax.set_xlabel("trial"); ax.set_ylabel("validation priority score"); ax.grid(alpha=.3); ax.legend(); plt.tight_layout(); plt.show()
    if len(done) >= N_TRIALS:
        study.stop()

if len([t for t in study.trials if real(t)]) < N_TRIALS:
    study.optimize(objective, timeout=TIMEOUT_MIN * 60, callbacks=[progress], gc_after_trial=True)
print("\ndone:", len([t for t in study.trials if real(t)]), "trained trials;", len(study.trials), "sampled in total")

## 6 · Search diagnostics

In [ ]:
trials = study.trials_dataframe(attrs=("number", "state", "value", "params", "user_attrs", "duration"))
trials = trials[trials["user_attrs_reason"].isna()] if "user_attrs_reason" in trials else trials
trials.to_csv(OUT_DIR / "optuna_trials.csv", index=False)
comp = trials[trials.state == "COMPLETE"].sort_values("value", ascending=False)
cols = [c for c in comp.columns if c in ("number", "value", "user_attrs_params", "user_attrs_val_auc", "user_attrs_val_recall_W", "user_attrs_val_recall_FC", "user_attrs_epochs") or c.startswith("params_")]
print(f"{len(trials)} trained trials: {len(comp)} complete, {(trials.state == 'PRUNED').sum()} pruned. Top 8 (validation):")
display(comp[cols].head(8).round(4))

best_P = dict(study.best_trial.params)
best_P["ratio"] = float(best_P["ratio"])
print("\nbest parameters:", json.dumps(best_P, indent=1), f"\nparameters of the network: {study.best_trial.user_attrs['params']:,}")
json.dump(best_P, open(OUT_DIR / "best_params.json", "w"), indent=1)

import optuna.visualization.matplotlib as ovm
for fn, title in ((ovm.plot_optimization_history, "optimization history"), (ovm.plot_param_importances, "parameter importance"), (ovm.plot_slice, "objective vs each parameter")):
    try:
        ax = fn(study)
        fig = ax.figure if hasattr(ax, "figure") else plt.gcf()
        fig.suptitle(title); plt.tight_layout(); plt.show()
    except Exception as e:
        print(f"({title} not drawn: {e})")

## 7 · Final training: e06 CRNN (reference) and tuned CRNN, several seeds (training curves shown here)

The same validation split and the same procedure for both, so the comparison is fair. Early stopping on validation AUC; the threshold is chosen on the validation probabilities of the seed ensemble; the 2025 test set is evaluated once.

In [ ]:
FINAL = {}
for tag, P in (("e06_crnn", BASELINE_P), ("tuned_crnn", best_P)):
    runs = []
    for s in FINAL_SEEDS:
        t0 = time.time()
        r = fit(P, s, FINAL_EPOCHS, FINAL_PATIENCE)
        thr_s = best_threshold(yv, r["pv"])
        print(f"{tag:11s} seed {s}: {r['epochs']:2d} epochs · val AUC {r['val_auc']:.3f} · val recall W {thr_s[2]:.3f} / FC {thr_s[3]:.3f} · {time.time() - t0:.0f} s")
        runs.append(r)
    pv, pe = np.mean([r["pv"] for r in runs], 0), np.mean([r["pe"] for r in runs], 0)
    FINAL[tag] = dict(P=P, runs=runs, pv=pv, pe=pe, thr=best_threshold(yv, pv)[0], params=count_params(runs[0]["model"]))

fig, axs = plt.subplots(2, 3, figsize=(15, 6.4), sharex=True)
for row, tag in enumerate(FINAL):
    for s, r in zip(FINAL_SEEDS, FINAL[tag]["runs"]):
        h = r["hist"]
        for ax, col in zip(axs[row], ("train_loss", "val_auc", "val_score")):
            ax.plot(h["epoch"], h[col], marker="o", ms=3, label=f"seed {s}")
    for ax, col in zip(axs[row], ("training loss", "validation AUC", "validation priority score")):
        ax.set_title(f"{tag}: {col}", fontsize=10); ax.grid(alpha=.3)
    axs[row][0].legend(fontsize=8)
for ax in axs[1]:
    ax.set_xlabel("epoch")
plt.tight_layout(); plt.savefig(OUT_DIR / "training_curves.png", dpi=120); plt.show()
pd.concat([r["hist"].assign(model=tag, seed=r["seed"]) for tag in FINAL for r in FINAL[tag]["runs"]]).to_csv(OUT_DIR / "training_histories.csv", index=False)

## 8 · Results on the test year 2025 (seed ensemble, threshold chosen on validation) with 95 % patient-clustered bootstrap intervals

In [ ]:
def full_metrics(y_true, p, thr):
    pred = (p >= thr).astype(int)
    r = classification_report(y_true, pred, labels=[0, 1], target_names=CLASSES, output_dict=True, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    rW, rFC = r["W"]["recall"], r["FC"]["recall"]
    return {"auc": roc_auc_score(y_true, p), "recall_W": rW, "precision_W": r["W"]["precision"], "f1_W": r["W"]["f1-score"],
            "recall_FC": rFC, "precision_FC": r["FC"]["precision"], "f1_FC": r["FC"]["f1-score"], "balanced_acc": balanced_accuracy_score(y_true, pred),
            "macro_f1": r["macro avg"]["f1-score"], "priority_score": priority(rW, rFC), "W_as_W": int(tn), "W_as_FC": int(fp), "FC_as_W": int(fn), "FC_as_FC": int(tp)}

def cluster_bootstrap(y_true, p, thr, groups, n=N_BOOT, seed=0):
    rng = np.random.default_rng(seed)
    ug = np.unique(groups); idx = {g: np.flatnonzero(groups == g) for g in ug}
    keys = ("auc", "recall_W", "precision_W", "f1_W", "recall_FC", "precision_FC", "f1_FC", "priority_score")
    out = {k: [] for k in keys}
    for _ in range(n):
        ii = np.concatenate([idx[g] for g in rng.choice(ug, len(ug))])
        if len(np.unique(y_true[ii])) < 2:
            continue
        m = full_metrics(y_true[ii], p[ii], thr)
        for k in keys:
            out[k].append(m[k])
    return {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in out.items()}

rows = []
for tag in FINAL:
    for name, thr in (("threshold 0.5", THRESHOLD), ("threshold tuned on validation", FINAL[tag]["thr"])):
        m = full_metrics(ye, FINAL[tag]["pe"], thr)
        ci = cluster_bootstrap(ye, FINAL[tag]["pe"], thr, ge) if "tuned" in name else {}
        rows.append(dict(model=tag, params=FINAL[tag]["params"], decision=name, threshold=round(thr, 3), **m,
                         **{f"{k}_ci": f"{lo:.3f}-{hi:.3f}" for k, (lo, hi) in ci.items()}))
res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / "results_test2025.csv", index=False)
show = ["model", "params", "decision", "threshold", "auc", "recall_W", "precision_W", "f1_W", "recall_FC", "precision_FC", "f1_FC", "balanced_acc", "priority_score"]
display(res[show].round(3))
print("95 % intervals (tuned threshold):")
display(res[res.decision.str.contains("tuned")][["model"] + [c for c in res.columns if c.endswith("_ci")]])

# confusion matrices
fig, axs = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, (_, r) in zip(axs, res.iterrows()):
    cm = np.array([[r.W_as_W, r.W_as_FC], [r.FC_as_W, r.FC_as_FC]]); cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, vmin=0, vmax=1, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}\n{cmn[i, j]:.1%}", ha="center", va="center", color="white" if cmn[i, j] > .5 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{r.model}\n{r.decision} ({r.threshold})", fontsize=9)
plt.tight_layout(); plt.savefig(OUT_DIR / "confusion_matrices.png", dpi=120); plt.show()

# ROC and the wheeze-recall / crackle-recall trade-off on the test year
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
for tag, c in zip(FINAL, ("tab:gray", "tab:red")):
    fpr, tpr, _ = roc_curve(ye, FINAL[tag]["pe"]); axs[0].plot(fpr, tpr, color=c, label=f"{tag} (AUC {roc_auc_score(ye, FINAL[tag]['pe']):.3f})")
    ths = np.linspace(0.01, 0.99, 99)
    rr = np.array([rec_at(ye, FINAL[tag]["pe"], t) for t in ths])
    axs[1].plot(rr[:, 1], rr[:, 0], color=c, label=tag)
    rW, rFC = rec_at(ye, FINAL[tag]["pe"], FINAL[tag]["thr"]); axs[1].plot(rFC, rW, "o", color=c, ms=9, label=f"{tag}: tuned threshold")
axs[0].plot([0, 1], [0, 1], "k:", lw=.8); axs[0].set_xlabel("1 - recall of W (false positive rate)"); axs[0].set_ylabel("recall of FC"); axs[0].set_title("ROC (FC = positive)"); axs[0].legend(); axs[0].grid(alpha=.3)
axs[1].axvline(FC_FLOOR, color="k", ls=":", lw=.8); axs[1].set_xlabel("recall of fine crackles (FC)"); axs[1].set_ylabel("recall of wheezes (W)")
axs[1].set_title("trade-off while the threshold moves (dotted = FC floor)"); axs[1].legend(fontsize=8); axs[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "roc_tradeoff.png", dpi=120); plt.show()

## 9 · Comparison with the state of the art

**Read this first.** The literature values (`SPRSound_SOTA_summary.md`) are for other tasks: normal vs abnormal (task 1-1), 7 classes (1-2) or 4 classes with Normal (Mamba-KAN), on other test splits.
No paper reports wheeze vs fine crackle alone, so **this is an orientation, not a like-for-like ranking**. To use the same language as the challenge, the W-vs-FC result is also written in its metrics:
SE = recall of W (the class we prioritise), SP = recall of FC, AS = (SE+SP)/2, HS = 2·SE·SP/(SE+SP), Score = (AS+HS)/2.

In [ ]:
def challenge(rW, rFC):
    AS, HS = (rW + rFC) / 2, 2 * rW * rFC / max(rW + rFC, 1e-9)
    return AS, HS, (AS + HS) / 2

ours = res[res.decision.str.contains("tuned")].set_index("model")
rows = []
for tag, label in (("e06_crnn", "This work: e06 CRNN (tuned threshold)"), ("tuned_crnn", "This work: Optuna-tuned CRNN")):
    r = ours.loc[tag]; AS, HS, sc = challenge(r.recall_W, r.recall_FC)
    rows.append({"model": label, "task": "W vs FC, 2025", "SE/recall W": r.recall_W, "SP/recall FC": r.recall_FC, "challenge-style Score": sc,
                 "W precision": r.precision_W, "W F1": r.f1_W, "FC precision": r.precision_FC, "FC F1": r.f1_FC})
lit = [
    ("Mamba-KAN (2026, lightweight)", "4 classes incl. Normal", None, None, None, 0.39, 0.42, 0.75, 0.58, 0.47, 0.48),
    ("ResNet18 fine-tuned (Chen 2022)", "1-1 normal vs abnormal", 0.89, 0.90, None, None, None, None, None, None, None),
    ("ResNet18 fine-tuned (Chen 2022)", "1-2 seven classes", 0.68, 0.94, None, None, None, None, None, None, None),
    ("Multi-stage CNN-Transformer (Shuvo 2025)", "1-1 normal vs abnormal", None, None, 0.9039, None, None, None, None, None, None),
    ("Multi-stage CNN-Transformer (Shuvo 2025)", "1-2 seven classes", None, None, 0.8448, None, None, None, None, None, None),
    ("Bi-SEResNet + DeiT (Wang 2025)", "1-1 normal vs abnormal", None, None, 0.8926, None, None, None, None, None, None),
]
for n, task, se, sp, score, wp, wf, cp, cf, wr, cr in lit:
    rows.append({"model": n, "task": task, "SE/recall W": wr if se is None else se, "SP/recall FC": cr if sp is None else sp, "challenge-style Score": score,
                 "W precision": wp, "W F1": wf, "FC precision": cp, "FC F1": cf})
sota = pd.DataFrame(rows)
sota.to_csv(OUT_DIR / "comparison_sota.csv", index=False)
display(sota.round(3).fillna("–"))
print("Literature rows: SE/SP of the ResNet18 rows are the abnormal-vs-normal sensitivity/specificity; for Mamba-KAN the columns 'recall W/FC' are the per-class recalls of wheeze and crackle (4-class problem). '–' = not reported.")

fig, ax = plt.subplots(figsize=(9, 3.8))
labels = ["W recall", "W precision", "W F1", "FC recall", "FC precision", "FC F1"]
sets = {"e06 CRNN": ours.loc["e06_crnn"], "Optuna CRNN": ours.loc["tuned_crnn"]}
vals = {k: [v.recall_W, v.precision_W, v.f1_W, v.recall_FC, v.precision_FC, v.f1_FC] for k, v in sets.items()}
vals["Mamba-KAN (4 cls, other data)"] = [0.47, 0.39, 0.42, 0.48, 0.75, 0.58]
w = 0.27
for i, (k, v) in enumerate(vals.items()):
    ax.bar(np.arange(6) + (i - 1) * w, v, w, label=k)
ax.set_xticks(np.arange(6)); ax.set_xticklabels(labels); ax.set_ylim(0, 1); ax.grid(axis="y", alpha=.3); ax.legend(fontsize=8)
ax.set_title("Per-class metrics vs the only published per-class SPRSound result (different task)", fontsize=9)
plt.tight_layout(); plt.savefig(OUT_DIR / "comparison_sota.png", dpi=120); plt.show()

## 10 · ONNX export of the tuned model (the best seed) and check

One network is exported (not the 3-seed ensemble, which would triple the parameters). The ONNX takes the log-mel `(batch, 1, 64, 95)` and returns the **probability of fine crackle**; the decision rule (`threshold.json`) is
"fine crackle if p ≥ threshold, otherwise wheeze". The threshold is chosen on the validation probabilities of that same network.

In [ ]:
import onnx, onnxruntime as ort

runs_t = FINAL["tuned_crnn"]["runs"]
pick = int(np.argmax([best_threshold(yv, r["pv"])[1] for r in runs_t]))          # best seed on VALIDATION (not on the test year)
best_run = runs_t[pick]
EXPORT_THR = best_threshold(yv, best_run["pv"])[0]

class WithSigmoid(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        return torch.sigmoid(self.m(x))

export_model = WithSigmoid(build_model(best_P)).eval()
export_model.m.load_state_dict({k: v.detach().cpu() for k, v in best_run["model"].state_dict().items()})
dummy = torch.zeros(1, 1, N_MELS, 95)
ONNX_PATH = OUT_DIR / "crnn_w_vs_fc.onnx"
kw = dict(input_names=["log_mel"], output_names=["p_fine_crackle"], dynamic_axes={"log_mel": {0: "batch"}, "p_fine_crackle": {0: "batch"}}, opset_version=17)
try:
    torch.onnx.export(export_model, dummy, str(ONNX_PATH), dynamo=False, **kw)
except TypeError:
    torch.onnx.export(export_model, dummy, str(ONNX_PATH), **kw)
onnx.checker.check_model(onnx.load(str(ONNX_PATH)))

sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
Xe_cpu = X[~tr][:, None].astype(np.float32)
p_onnx = np.concatenate([sess.run(None, {"log_mel": Xe_cpu[i:i + 256]})[0] for i in range(0, len(Xe_cpu), 256)]).ravel()
with torch.no_grad():
    p_torch = np.concatenate([export_model(torch.from_numpy(Xe_cpu[i:i + 256])).numpy() for i in range(0, len(Xe_cpu), 256)])
max_diff = float(np.abs(p_onnx - p_torch).max())
print(f"seed {FINAL_SEEDS[pick]} exported -> {ONNX_PATH.name} ({ONNX_PATH.stat().st_size / 1e3:.0f} kB) · max |ONNX - PyTorch| over {len(p_onnx)} test events: {max_diff:.2e}")
assert max_diff < 1e-2, "ONNX output differs from PyTorch"
m_onnx = full_metrics(ye, p_onnx, EXPORT_THR)
print(f"ONNX model on the test year at threshold {EXPORT_THR:.3f}: AUC {m_onnx['auc']:.3f} · recall W {m_onnx['recall_W']:.3f} (precision {m_onnx['precision_W']:.3f}) · recall FC {m_onnx['recall_FC']:.3f} (precision {m_onnx['precision_FC']:.3f})")
t0 = time.time()
for _ in range(100):
    sess.run(None, {"log_mel": Xe_cpu[:1]})
LATENCY_MS = (time.time() - t0) / 100 * 1e3
json.dump({"input": "log_mel (batch,1,64,95) float32: 3 s window at 4 kHz, n_fft 512, hop 128, 64 mel bands 50-2000 Hz, 10*log10 power, normalised per sample (zero mean, unit std)",
           "output": "p_fine_crackle in [0,1]", "threshold": EXPORT_THR, "rule": "fine crackle if p >= threshold else wheeze", "seed": FINAL_SEEDS[pick], "parameters": best_P},
          open(OUT_DIR / "threshold.json", "w"), indent=1)

## 11 · Model report and drawing

In [ ]:
# per-layer table with output shape, parameters and multiply-accumulates (batch of 1)
net = export_model.m.eval()
rows, hooks = [], []
def hook(name):
    def f(m, i, o):
        o0 = o[0] if isinstance(o, tuple) else o
        p = sum(q.numel() for q in m.parameters())
        if isinstance(m, nn.Conv2d):
            macs = o0.numel() * m.in_channels * m.kernel_size[0] * m.kernel_size[1]
        elif isinstance(m, nn.Linear):
            macs = m.in_features * m.out_features
        elif isinstance(m, nn.GRU):
            T, h, macs = i[0].shape[1], m.hidden_size, 0
            for l in range(m.num_layers):
                inp = m.input_size if l == 0 else 2 * h
                macs += 2 * T * 3 * (inp * h + h * h)
        else:
            macs = 0
        rows.append(dict(layer=name, type=type(m).__name__, output_shape=tuple(o0.shape[1:]), params=p, MACs=int(macs)))
    return f
for name, m in net.named_modules():
    if len(list(m.children())) == 0 or isinstance(m, nn.GRU):
        hooks.append(m.register_forward_hook(hook(name)))
with torch.no_grad():
    net(dummy)
for h in hooks:
    h.remove()
layers = pd.DataFrame(rows)
layers.to_csv(OUT_DIR / "model_layers.csv", index=False)
display(layers)
n_par, macs = count_params(net), int(layers.MACs.sum())
print(f"parameters {n_par:,} · MACs per 3-s event {macs / 1e6:.1f} M · fp32 weights {n_par * 4 / 1e6:.2f} MB · ONNX file {ONNX_PATH.stat().st_size / 1e3:.0f} kB · CPU latency (onnxruntime, batch 1) {LATENCY_MS:.1f} ms")

# drawing of the architecture, sizes from the hooks above
from matplotlib.patches import FancyBboxPatch
ch, shp = channels_of(best_P), {r["layer"]: r["output_shape"] for r in rows}
pool_names = [f"conv.{4 * i + 3}" for i in range(3)]
pblk = [sum(p.numel() for p in net.conv[4 * i:4 * i + 4].parameters()) for i in range(3)]
boxes = [("Input\nlog-mel\n1 x %d x 95\n(freq x time)" % N_MELS, "#e0f2fe", 0)]
for i in range(3):
    boxes.append((f"Conv 3x3, {ch[i]} ch\nBatchNorm, ReLU\nMaxPool {'2x2' if i < 2 else '2x1'}\n-> {'x'.join(map(str, shp[pool_names[i]]))}", "#dbeafe", pblk[i]))
T_, F_ = shp[pool_names[2]][2], shp[pool_names[2]][1]
boxes.append((f"Reshape\n{T_} time steps x {ch[2] * shp[pool_names[2]][1]} features", "#f1f5f9", 0))
boxes.append((f"Bi-GRU x{best_P['gru_layers']}\nhidden {best_P['hidden']} per direction\n-> {T_} x {2 * best_P['hidden']}", "#fde68a", sum(p.numel() for p in net.rnn.parameters())))
boxes.append((f"{best_P['pool'].capitalize()} over time\n-> {2 * best_P['hidden']}", "#f1f5f9", 0))
boxes.append((f"Dropout {best_P['dropout']:.2f}\nLinear {best_P['units']}, ReLU\nLinear 1", "#fecaca", sum(p.numel() for p in net.head.parameters())))
boxes.append((f"Sigmoid\np(fine crackle)\nW if p < {EXPORT_THR:.2f}", "#bbf7d0", 0))
per_row = 5
fig, ax = plt.subplots(figsize=(15, 5.2)); ax.set_xlim(0, per_row * 3); ax.set_ylim(0, 2 * 2.6); ax.axis("off")
pos = []
for k, (txt, col, npar) in enumerate(boxes):
    r, c = divmod(k, per_row)
    x, yb = 0.3 + c * 3, (1 - r) * 2.6 + 0.5
    ax.add_patch(FancyBboxPatch((x, yb), 2.3, 1.7, boxstyle="round,pad=0.05", fc=col, ec="#334155"))
    ax.text(x + 1.15, yb + 0.95, txt, ha="center", va="center", fontsize=8.5)
    if npar:
        ax.text(x + 1.15, yb + 0.12, f"{npar:,} params", ha="center", va="center", fontsize=7.5, color="#475569", style="italic")
    pos.append((x, yb))
for k in range(len(boxes) - 1):
    (x0, y0), (x1, y1) = pos[k], pos[k + 1]
    if y0 == y1:
        ax.annotate("", xy=(x1 - 0.05, y1 + 0.85), xytext=(x0 + 2.35, y0 + 0.85), arrowprops=dict(arrowstyle="->", color="#334155"))
    else:
        ym = y0 - 0.45                                                   # elbow connector between the two rows, drawn in the gap
        ax.plot([x0 + 1.15, x0 + 1.15, x1 + 1.15], [y0 - 0.05, ym, ym], color="#334155", lw=1)
        ax.annotate("", xy=(x1 + 1.15, y1 + 1.75), xytext=(x1 + 1.15, ym), arrowprops=dict(arrowstyle="->", color="#334155"))
ax.set_title(f"Tuned CRNN for wheeze (W) vs fine crackle (FC): {n_par:,} parameters, {macs / 1e6:.1f} M MACs per event", fontsize=11)
plt.tight_layout(); plt.savefig(OUT_DIR / "model_diagram.png", dpi=140); plt.show()
print("For an interactive graph of the ONNX file open https://netron.app and drop crnn_w_vs_fc.onnx on it.")

In [ ]:
# Short model report (markdown + CSV)
r_t, r_b = ours.loc["tuned_crnn"], ours.loc["e06_crnn"]
ci = res[(res.model == "tuned_crnn") & res.decision.str.contains("tuned")].iloc[0]
report = f"""# Tuned CRNN for wheeze vs fine crackle (SPRSound, test year {HELD_OUT})

**Task.** Event-level wheeze (W) vs fine crackle (FC). Train 2022-2024, test 2025, validation = {int(VAL_FRAC * 100)} % of the training patients. Priority: wheeze recall (weight {W_WEIGHT}) without letting FC recall fall under {FC_FLOOR}.

**Model.** Log-mel (1 x {N_MELS} x 95) -> 3 x [Conv3x3-BN-ReLU-MaxPool] with channels {channels_of(best_P)} -> bidirectional GRU (hidden {best_P['hidden']}, {best_P['gru_layers']} layer(s)) -> {best_P['pool']} pooling over time -> Dropout {best_P['dropout']:.2f} -> Linear {best_P['units']} -> ReLU -> Linear 1 -> sigmoid.
{n_par:,} parameters, {macs / 1e6:.1f} M multiply-accumulates per 3-s event, ONNX file {ONNX_PATH.stat().st_size / 1e3:.0f} kB, CPU latency {LATENCY_MS:.1f} ms (onnxruntime, batch 1).

**Training.** AdamW (lr {best_P['lr']:.2e}, weight decay {best_P['wd']:.2e}), batch {best_P['batch']}, FC loss weight x{best_P['w_pos_weight_factor']:.2f} of the balanced weight, SpecAugment-style masks and circular shift, early stopping on validation AUC. Search: Optuna TPE, {len([t for t in study.trials if real(t)])} trials, {PARAM_MIN:,}-{PARAM_MAX:,} parameters.

**Test results 2025 ({len(FINAL_SEEDS)}-seed ensemble, threshold {FINAL['tuned_crnn']['thr']:.3f} chosen on validation)** - tuned CRNN vs the e06 CRNN:

| metric | tuned CRNN | e06 CRNN | 95 % CI tuned |
|---|---:|---:|---|
| AUC | {r_t.auc:.3f} | {r_b.auc:.3f} | {ci.auc_ci} |
| recall W | {r_t.recall_W:.3f} | {r_b.recall_W:.3f} | {ci.recall_W_ci} |
| precision W | {r_t.precision_W:.3f} | {r_b.precision_W:.3f} | {ci.precision_W_ci} |
| recall FC | {r_t.recall_FC:.3f} | {r_b.recall_FC:.3f} | {ci.recall_FC_ci} |
| precision FC | {r_t.precision_FC:.3f} | {r_b.precision_FC:.3f} | {ci.precision_FC_ci} |
| priority score | {r_t.priority_score:.3f} | {r_b.priority_score:.3f} | {ci.priority_score_ci} |

The exported network (seed {FINAL_SEEDS[pick]}, threshold {EXPORT_THR:.3f}): AUC {m_onnx['auc']:.3f}, recall W {m_onnx['recall_W']:.3f}, recall FC {m_onnx['recall_FC']:.3f}.

**Caveats.** One test year with few patients (see the intervals); the threshold, architecture and loss weight were chosen on validation data only; literature numbers are for different tasks (see the comparison table).
"""
(OUT_DIR / "report_model.md").write_text(report, encoding="utf-8")
print(report)

## 12 · Save

In [ ]:
pack = Path("/content/gate2_crnn_optuna_results"); pack.mkdir(exist_ok=True)
for f in ("report_model.md", "results_test2025.csv", "comparison_sota.csv", "optuna_trials.csv", "best_params.json", "training_histories.csv", "model_layers.csv", "threshold.json",
          "crnn_w_vs_fc.onnx", "model_diagram.png", "training_curves.png", "confusion_matrices.png", "roc_tradeoff.png", "comparison_sota.png"):
    if (OUT_DIR / f).exists():
        shutil.copy2(OUT_DIR / f, pack / f)
zip_path = shutil.make_archive(str(pack), "zip", pack)
print("saved:", sorted(p.name for p in pack.iterdir()), "\nzip:", zip_path)

In [ ]:
# Optional: send the zip to YOUR computer (browser Colab; the VS Code extension shows a fallback link). Run only if you want the files locally.
import base64
from IPython.display import HTML
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("files.download not available here:", e)
    b64 = base64.b64encode(Path(zip_path).read_bytes()).decode()
    display(HTML(f'<a download="{Path(zip_path).name}" href="data:application/zip;base64,{b64}">Download {Path(zip_path).name}</a>'))